In [1]:
# A0 — Install TF 2.20
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [2]:
# A1 — Imports and config
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc
import re
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

CFG = {
    "batch_files": 64,   # 如果内存不够就改 32
    "proxy_reduce": "max",
    "verbose": True,
}

print("TensorFlow:", tf.__version__)
print("BASE exists:", BASE.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())

TensorFlow: 2.20.0
BASE exists: True
MODEL_DIR exists: True


In [3]:
# A2 — Load competition metadata
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
soundscape_labels["primary_label"] = soundscape_labels["primary_label"].astype(str)

def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [t.strip() for t in str(x).split(";") if t.strip()]

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {
            "file_id": None,
            "site": None,
            "date": pd.NaT,
            "time_utc": None,
            "hour_utc": -1,
            "month": -1,
        }
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {
        "file_id": file_id,
        "site": site,
        "date": dt,
        "time_utc": hms,
        "hour_utc": int(hms[:2]),
        "month": int(dt.month) if pd.notna(dt) else -1,
    }

def union_labels(series):
    return sorted(set(lbl for x in series for lbl in parse_soundscape_labels(x)))

sc_clean = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)

sc_clean["start_sec"] = pd.to_timedelta(sc_clean["start"]).dt.total_seconds().astype(int)
sc_clean["end_sec"] = pd.to_timedelta(sc_clean["end"]).dt.total_seconds().astype(int)
sc_clean["row_id"] = sc_clean["filename"].str.replace(".ogg", "", regex=False) + "_" + sc_clean["end_sec"].astype(str)

meta = sc_clean["filename"].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

windows_per_file = sc_clean.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean["file_fully_labeled"] = sc_clean["filename"].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for i, labels in enumerate(sc_clean["label_list"]):
    idxs = [label_to_idx[lbl] for lbl in labels if lbl in label_to_idx]
    if idxs:
        Y_SC[i, idxs] = 1

full_truth = (
    sc_clean[sc_clean["file_fully_labeled"]]
    .sort_values(["filename", "end_sec"])
    .reset_index(drop=False)
)

Y_FULL_TRUTH = Y_SC[full_truth["index"].to_numpy()]

print("full files:", len(full_files))
print("trusted windows:", len(full_truth))
print("active classes:", int((Y_FULL_TRUTH.sum(axis=0) > 0).sum()))

full files: 59
trusted windows: 708
active classes: 71


In [4]:
# A3 — Load Perch + mapping/proxy
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)

NO_LABEL_INDEX = len(bc_labels)

taxonomy = taxonomy.copy()
taxonomy["scientific_name_lookup"] = taxonomy["scientific_name"]

bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})
mapping = taxonomy.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup",
    how="left"
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ].copy()
    return genus, hits

proxy_map = {}
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            "bc_indices": hits["bc_index"].astype(int).tolist(),
            "genus": genus,
        }

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys()
    if CLASS_NAME_MAP.get(t) in PROXY_TAXA
])

selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

print("mapped:", int(MAPPED_MASK.sum()), "/", N_CLASSES)
print("proxy targets:", len(SELECTED_PROXY_TARGETS))

mapped: 203 / 234
proxy targets: 3


In [5]:
# A4 — Perch inference helpers
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr} in {path}; expected {SR}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y

def infer_perch_with_embeddings(paths, batch_files=64, verbose=True, proxy_reduce="max"):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)

    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc="Perch batches")

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)

        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row
        x_pos = 0

        for path in batch_paths:
            y = read_soundscape_60s(path)
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            stem = path.stem

            row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])

            x_pos += N_WINDOWS
            write_row += N_WINDOWS

        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs["label"].numpy().astype(np.float32, copy=False)
        emb = outputs["embedding"].numpy().astype(np.float32, copy=False)

        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            proxy_score = sub.max(axis=1) if proxy_reduce == "max" else sub.mean(axis=1)
            scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)

        del x, outputs, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        "row_id": row_ids,
        "filename": filenames,
        "site": sites,
        "hour_utc": hours,
    })
    return meta_df, scores, embeddings

In [6]:
# A5 — Fit prior tables
EPS = 1e-4

def fit_prior_tables(full_truth_df, y_true):
    df = full_truth_df.copy().reset_index(drop=True)
    global_prob = y_true.mean(axis=0).astype(np.float32)

    def group_prob(col_names):
        tmp = pd.DataFrame(y_true, columns=PRIMARY_LABELS)
        grp = pd.concat([df[col_names], tmp], axis=1)
        return grp.groupby(col_names)[PRIMARY_LABELS].mean().astype(np.float32)

    tables = {
        "global": global_prob,
        "site": group_prob(["site"]),
        "hour": group_prob(["hour_utc"]),
        "month": group_prob(["month"]),
        "site_hour": group_prob(["site", "hour_utc"]),
    }
    return tables

PRIOR_TABLES = fit_prior_tables(full_truth, Y_FULL_TRUTH)

print("Prior tables ready.")
print("site groups:", len(PRIOR_TABLES["site"]))
print("hour groups:", len(PRIOR_TABLES["hour"]))
print("month groups:", len(PRIOR_TABLES["month"]))
print("site_hour groups:", len(PRIOR_TABLES["site_hour"]))

Prior tables ready.
site groups: 8
hour groups: 13
month groups: 7
site_hour groups: 23


In [7]:
# A6 — Build probe artifact from trusted full files
full_train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]

meta_full_probe, scores_full_raw_probe, emb_full_probe = infer_perch_with_embeddings(
    full_train_paths,
    batch_files=CFG["batch_files"],
    verbose=CFG["verbose"],
    proxy_reduce=CFG["proxy_reduce"],
)

assert len(meta_full_probe) == len(full_truth)

scaler_probe = StandardScaler()
X_full_scaled = scaler_probe.fit_transform(emb_full_probe)

PCA_DIM = 128
pca_probe = PCA(n_components=PCA_DIM, random_state=42)
X_full_pca = pca_probe.fit_transform(X_full_scaled)

active_class_idx = np.where(Y_FULL_TRUTH.sum(axis=0) > 0)[0].astype(np.int32)

coef_mat = np.zeros((len(active_class_idx), PCA_DIM), dtype=np.float32)
intercept_vec = np.zeros(len(active_class_idx), dtype=np.float32)

for k, j in enumerate(tqdm(active_class_idx, desc="Training simple probes")):
    yj = Y_FULL_TRUTH[:, j]
    clf = LogisticRegression(
        C=1.0,
        max_iter=300,
        solver="liblinear",
        class_weight="balanced",
        random_state=42,
    )
    clf.fit(X_full_pca, yj)
    coef_mat[k] = clf.coef_[0].astype(np.float32)
    intercept_vec[k] = np.float32(clf.intercept_[0])

print("emb_full_probe:", emb_full_probe.shape)
print("X_full_pca:", X_full_pca.shape)
print("trained probe classes:", len(active_class_idx))

Perch batches:   0%|          | 0/1 [00:00<?, ?it/s]

I0000 00:00:1776518296.804778      68 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Training simple probes:   0%|          | 0/71 [00:00<?, ?it/s]

emb_full_probe: (708, 1536)
X_full_pca: (708, 128)
trained probe classes: 71


In [8]:
# A6.5 — Build temporal-lite file-level features from 12-window raw logits

from sklearn.linear_model import LogisticRegression

# scores_full_raw_probe: shape (708, 234)
# full files: 59
# each file has 12 windows
n_full_files = len(full_files)

assert scores_full_raw_probe.shape[0] == n_full_files * N_WINDOWS

# reshape to (n_files, 12, 234)
scores_full_seq = scores_full_raw_probe.reshape(n_full_files, N_WINDOWS, N_CLASSES)

# build file-level labels: if any of 12 windows is positive, file label = 1
Y_FILE_TRUTH = Y_FULL_TRUTH.reshape(n_full_files, N_WINDOWS, N_CLASSES).max(axis=1).astype(np.uint8)

# temporal-lite per-class features:
# mean, max, std, last-first  => 4 features per class
feat_mean = scores_full_seq.mean(axis=1)                         # (n_files, 234)
feat_max = scores_full_seq.max(axis=1)                          # (n_files, 234)
feat_std = scores_full_seq.std(axis=1)                          # (n_files, 234)
feat_delta = scores_full_seq[:, -1, :] - scores_full_seq[:, 0, :]  # (n_files, 234)

print("scores_full_seq:", scores_full_seq.shape)
print("Y_FILE_TRUTH:", Y_FILE_TRUTH.shape)
print("feat_mean:", feat_mean.shape)

scores_full_seq: (59, 12, 234)
Y_FILE_TRUTH: (59, 234)
feat_mean: (59, 234)


In [9]:
# A6.6 — Train temporal-lite logistic heads (4 features per class)

temp_active_class_idx = np.where(Y_FILE_TRUTH.sum(axis=0) > 0)[0].astype(np.int32)

# For each active class j, train a 4-feature logistic regression:
# X_j = [mean_j, max_j, std_j, delta_j]
temp_coef_mat = np.zeros((len(temp_active_class_idx), 4), dtype=np.float32)
temp_intercept_vec = np.zeros(len(temp_active_class_idx), dtype=np.float32)

for k, j in enumerate(tqdm(temp_active_class_idx, desc="Training temporal-lite heads")):
    yj = Y_FILE_TRUTH[:, j]

    Xj = np.stack([
        feat_mean[:, j],
        feat_max[:, j],
        feat_std[:, j],
        feat_delta[:, j],
    ], axis=1).astype(np.float32)

    clf = LogisticRegression(
        C=1.0,
        max_iter=300,
        solver="liblinear",
        class_weight="balanced",
        random_state=42,
    )
    clf.fit(Xj, yj)

    temp_coef_mat[k] = clf.coef_[0].astype(np.float32)
    temp_intercept_vec[k] = np.float32(clf.intercept_[0])

print("temp_active_class_idx:", temp_active_class_idx.shape)
print("temp_coef_mat:", temp_coef_mat.shape)
print("temp_intercept_vec:", temp_intercept_vec.shape)

Training temporal-lite heads:   0%|          | 0/71 [00:00<?, ?it/s]

temp_active_class_idx: (71,)
temp_coef_mat: (71, 4)
temp_intercept_vec: (71,)


In [10]:
# A7 — Serialize all artifacts, including temporal-lite heads
def serialize_prior_table(df):
    if isinstance(df.index, pd.MultiIndex):
        keys = list(df.index)
    else:
        keys = list(df.index.tolist())
    vals = df.to_numpy(dtype=np.float32)
    return keys, vals

site_keys, site_vals = serialize_prior_table(PRIOR_TABLES["site"])
hour_keys, hour_vals = serialize_prior_table(PRIOR_TABLES["hour"])
month_keys, month_vals = serialize_prior_table(PRIOR_TABLES["month"])
site_hour_keys, site_hour_vals = serialize_prior_table(PRIOR_TABLES["site_hour"])

artifact = {
    "primary_labels": PRIMARY_LABELS,

    # prior
    "global_prob": PRIOR_TABLES["global"].astype(np.float32),
    "site_keys": site_keys,
    "site_vals": site_vals.astype(np.float32),
    "hour_keys": hour_keys,
    "hour_vals": hour_vals.astype(np.float32),
    "month_keys": month_keys,
    "month_vals": month_vals.astype(np.float32),
    "site_hour_keys": site_hour_keys,
    "site_hour_vals": site_hour_vals.astype(np.float32),

    # probe
    "scaler_mean": scaler_probe.mean_.astype(np.float32),
    "scaler_scale": scaler_probe.scale_.astype(np.float32),
    "pca_mean": pca_probe.mean_.astype(np.float32),
    "pca_components": pca_probe.components_.astype(np.float32),
    "active_class_idx": active_class_idx.astype(np.int32),
    "coef_mat": coef_mat.astype(np.float32),
    "intercept_vec": intercept_vec.astype(np.float32),

    # temporal-lite
    "temp_active_class_idx": temp_active_class_idx.astype(np.int32),
    "temp_coef_mat": temp_coef_mat.astype(np.float32),
    "temp_intercept_vec": temp_intercept_vec.astype(np.float32),
}

with open("offline_artifacts_v4_temporal.pkl", "wb") as f:
    pickle.dump(artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved offline_artifacts_v4_temporal.pkl")
print("File exists:", Path("offline_artifacts_v4_temporal.pkl").exists())
print("File size (MB):", Path("offline_artifacts_v4_temporal.pkl").stat().st_size / 1024 / 1024)

Saved offline_artifacts_v4_temporal.pkl
File exists: True
File size (MB): 0.8541507720947266


In [11]:
# A8 — Quick verification
with open("offline_artifacts_v4_temporal.pkl", "rb") as f:
    chk = pickle.load(f)

for k in [
    "global_prob", "site_vals", "hour_vals", "month_vals", "site_hour_vals",
    "scaler_mean", "scaler_scale", "pca_mean", "pca_components",
    "active_class_idx", "coef_mat", "intercept_vec",
    "temp_active_class_idx", "temp_coef_mat", "temp_intercept_vec",
]:
    v = chk[k]
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

global_prob (234,) float32
site_vals (8, 234) float32
hour_vals (13, 234) float32
month_vals (7, 234) float32
site_hour_vals (23, 234) float32
scaler_mean (1536,) float32
scaler_scale (1536,) float32
pca_mean (1536,) float32
pca_components (128, 1536) float32
active_class_idx (71,) int32
coef_mat (71, 128) float32
intercept_vec (71,) float32
temp_active_class_idx (71,) int32
temp_coef_mat (71, 4) float32
temp_intercept_vec (71,) float32
